# 🔬 Research Agent Pro — HITL-Powered Agentic AI
### Human-in-the-Loop Middleware with LangGraph + SqliteSaver + Ollama

> **Architecture**: Every tool call is intercepted before execution.  
> The human can **approve ✅**, **edit ✏️**, or **reject ❌** — the graph resumes from the exact checkpoint.

```
User Query → LLM (ReAct) → INTERRUPT → Human Review → Command(resume) → Tool Executes → LLM Answer
```

| Layer | Technology |
|---|---|
| Framework | LangChain 1.x |
| Graph Engine | LangGraph + SqliteSaver |
| LLM Backend | Ollama (local) |
| Package Manager | uv |
| Research Tools | DuckDuckGo · Arxiv · Wikipedia |
| Interface | CLI + real-time streaming |


## 📦 Step 1 — Install Dependencies
Run this command in your terminal (requires [uv](https://docs.astral.sh/uv/)):

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  uv add command — run in terminal (NOT inside the notebook)     │
# └─────────────────────────────────────────────────────────────────┘
#
# uv add langchain langchain-core langchain-ollama langchain-community \
#         langgraph duckduckgo-search wikipedia arxiv
#
# One-liner (copy & paste):
# uv add langchain langchain-core langchain-ollama langchain-community langgraph duckduckgo-search wikipedia arxiv

# ── To install inside the notebook instead, uncomment below ──────────────────
# import subprocess
# subprocess.run([
#     "uv", "add",
#     "langchain", "langchain-core", "langchain-ollama", "langchain-community",
#     "langgraph", "duckduckgo-search", "wikipedia", "arxiv"
# ], check=True)

print("✅ Copy the uv add command above and run it in your terminal.")
print()
print("Full command:")
print("uv add langchain langchain-core langchain-ollama langchain-community langgraph duckduckgo-search wikipedia arxiv")


## 📥 Step 2 — Imports

In [ ]:
import uuid
import json
from typing import Annotated, TypedDict, Literal

# ── LangChain core ────────────────────────────────────────────────────────────
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)
from langchain_core.tools import BaseTool

# ── LangChain Ollama (local LLM — no API key required) ───────────────────────
from langchain_ollama import ChatOllama

# ── LangChain Community Tools ─────────────────────────────────────────────────
from langchain_community.tools import (
    DuckDuckGoSearchRun,
    ArxivQueryRun,
    WikipediaQueryRun,
)
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

print("✅ All imports loaded successfully.")


## ⚙️ Step 3 — Configuration
> 💡 Make sure Ollama is running: `ollama serve`  
> Pull a model first: `ollama pull llama3.2`  (or `mistral`, `gemma2`, `qwen2.5`, etc.)


In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "llama3.2"               # Change to your pulled model
OLLAMA_BASE_URL = "http://localhost:11434"
DB_PATH         = "research_agent_hitl.db" # SqliteSaver checkpoint database

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a precise research assistant with access to three tools:
- duckduckgo_search : Real-time web search for current events and general queries
- arxiv             : Academic papers, scientific research topics
- wikipedia         : Encyclopedic background, definitions, historical context

Use the most relevant tool for each sub-question. Always cite sources.
Think step-by-step. If one tool is insufficient, call another.
"""

print(f"🤖 Model    : {OLLAMA_MODEL}")
print(f"💾 Database : {DB_PATH}")
print(f"🌐 Ollama   : {OLLAMA_BASE_URL}")


## 🛠️ Step 4 — Research Tools

In [ ]:
# ── DuckDuckGo ───────────────────────────────────────────────────────────────
ddg_tool  = DuckDuckGoSearchRun(name="duckduckgo_search")

# ── Arxiv ─────────────────────────────────────────────────────────────────────
arxiv_wrapper = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=800)
arxiv_tool    = ArxivQueryRun(api_wrapper=arxiv_wrapper, name="arxiv")

# ── Wikipedia ─────────────────────────────────────────────────────────────────
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=800)
wiki_tool    = WikipediaQueryRun(api_wrapper=wiki_wrapper, name="wikipedia")

# ── Registry ──────────────────────────────────────────────────────────────────
TOOLS: list[BaseTool]         = [ddg_tool, arxiv_tool, wiki_tool]
TOOLS_BY_NAME: dict[str, BaseTool] = {t.name: t for t in TOOLS}

print("🛠️  Registered tools:")
for t in TOOLS:
    print(f"   • {t.name:25s}")


## 🧠 Step 5 — LLM Initialization (Ollama local)

In [ ]:
llm = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
    streaming=True,
)

# Bind tools — enables structured tool_call output from the LLM
llm_with_tools = llm.bind_tools(TOOLS)

print(f"✅ LLM ready      : {OLLAMA_MODEL}")
print(f"🔗 Tool-bound to  : {[t.name for t in TOOLS]}")


## 📐 Step 6 — Agent State Schema

In [ ]:
class AgentState(TypedDict):
    """Shared mutable state passed across every graph node."""
    # Full conversation: HumanMessage, AIMessage, ToolMessage
    messages: Annotated[list[BaseMessage], add_messages]
    # Tool calls proposed by the LLM — awaiting human decision
    pending_tool_calls: list
    # Human decisions: list of {action, edited_args?, reason?}
    tool_decisions: list

print("✅ AgentState defined.")
print("   • messages           — full conversation (reducer: add_messages)")
print("   • pending_tool_calls — tool calls awaiting approval")
print("   • tool_decisions     — human decisions: approve / edit / reject")


## 🏗️ Step 7 — Graph Nodes

### Node Map
```
START
  │
  ▼
agent_node ──(tool_calls?)──► hitl_node ◄── INTERRUPT fires here
  ▲                                │
  │                                ▼
  └────────────────── tool_executor_node
  │
(no tool_calls → END)
```


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# NODE 1 │ agent_node
# Invokes the LLM. If the response has tool_calls → route to hitl_node.
# If the response is a final answer → route to END.
# ═════════════════════════════════════════════════════════════════════════════
def agent_node(state: AgentState) -> dict:
    """LLM reasoning step — emits a final answer or structured tool_calls."""
    messages  = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response: AIMessage = llm_with_tools.invoke(messages)
    return {
        "messages":           [response],
        "pending_tool_calls": response.tool_calls or [],
    }


# ═════════════════════════════════════════════════════════════════════════════
# NODE 2 │ hitl_node  ◄── INTERRUPT fires here
# interrupt() serialises the pending tool calls to SqliteSaver and suspends.
# Awaits Command(resume={"decisions": [...]}) from the human.
# ═════════════════════════════════════════════════════════════════════════════
def hitl_node(state: AgentState) -> dict:
    """
    Human-in-the-Loop gate.

    Key rules:
    • NEVER wrap interrupt() in try/except — it raises GraphInterrupt internally.
    • Payloads must be JSON-serialisable (no functions or class instances).
    • The node re-runs on resume — keep any side effects before this idempotent.
    """
    tool_calls = state.get("pending_tool_calls", [])

    # Build a clean, serialisable display payload
    display_payload = [
        {
            "index":   i + 1,
            "tool":    tc["name"],
            "args":    tc["args"],
            "call_id": tc["id"],
        }
        for i, tc in enumerate(tool_calls)
    ]

    # ── INTERRUPT: execution pauses here ─────────────────────────────────────
    human_response = interrupt({
        "message": "⏸️  HITL INTERRUPT — Review the following tool call(s):",
        "pending_tool_calls": display_payload,
        "instructions": (
            "Return: {'decisions': ["
            "{'action': 'approve'|'edit'|'reject', "
            "'edited_args': {...} (only if edit), "
            "'reason': str (only if reject)}]}"
        ),
    })
    # ── Execution resumes here after Command(resume=...) ─────────────────────

    decisions = human_response.get("decisions", [])
    if not decisions:
        # Default: approve all if human returns empty payload
        decisions = [{"action": "approve"} for _ in tool_calls]

    return {"tool_decisions": decisions}


# ═════════════════════════════════════════════════════════════════════════════
# NODE 3 │ tool_executor_node
# Processes each tool call according to its matching human decision:
#   approve → run tool as-is
#   edit    → run tool with human-supplied args
#   reject  → inject ToolMessage with rejection reason (tool skipped)
# ═════════════════════════════════════════════════════════════════════════════
def tool_executor_node(state: AgentState) -> dict:
    """Execute (or skip) each tool call based on human decisions."""
    tool_calls    = state.get("pending_tool_calls", [])
    decisions     = state.get("tool_decisions", [])
    tool_messages: list[ToolMessage] = []

    for i, tc in enumerate(tool_calls):
        decision = decisions[i] if i < len(decisions) else {"action": "approve"}
        action   = decision.get("action", "approve")

        if action == "reject":
            reason = decision.get("reason", "No reason provided.")
            print(f"   ❌ REJECTED  [{tc['name']}] — {reason}")
            tool_messages.append(ToolMessage(
                content=f"[REJECTED by human] {reason}",
                tool_call_id=tc["id"],
            ))

        elif action in ("approve", "edit"):
            run_args = (
                decision.get("edited_args", tc["args"])
                if action == "edit" else tc["args"]
            )
            tag = "✏️  EDITED" if action == "edit" else "✅ APPROVED"
            print(f"   {tag}  [{tc['name']}]  args={run_args}")

            tool = TOOLS_BY_NAME.get(tc["name"])
            if tool is None:
                result = f"Error: unknown tool '{tc['name']}'"
            else:
                try:
                    result = tool.invoke(run_args)
                except Exception as exc:
                    result = f"Tool error: {exc}"

            tool_messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tc["id"],
            ))

    return {
        "messages":           tool_messages,
        "pending_tool_calls": [],
        "tool_decisions":     [],
    }


# ── ROUTER ────────────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["hitl", "__end__"]:
    """After agent_node: if tool calls exist → HITL gate; else → END."""
    return "hitl" if state.get("pending_tool_calls") else "__end__"


print("✅ All nodes defined:")
print("   • agent_node          — LLM reasoning (ReAct)")
print("   • hitl_node           — INTERRUPT gate (approve / edit / reject)")
print("   • tool_executor_node  — conditional tool execution")
print("   • should_continue     — router: hitl | __end__")


## 🕸️ Step 8 — Build & Compile LangGraph with SqliteSaver

`SqliteSaver` is the persistent checkpointer — it stores the full graph state  
(messages, pending tool calls, decisions) to disk at every node transition.  
This enables **cross-session resume**: close your notebook, come back, resume.


In [ ]:
# ── Graph definition ─────────────────────────────────────────────────────────
graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent",         agent_node)
graph_builder.add_node("hitl",          hitl_node)
graph_builder.add_node("tool_executor", tool_executor_node)

# ── Edges ─────────────────────────────────────────────────────────────────────
graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"hitl": "hitl", "__end__": END},
)

graph_builder.add_edge("hitl",          "tool_executor")
graph_builder.add_edge("tool_executor", "agent")  # loop back after tool exec

# ── Compile with SqliteSaver ──────────────────────────────────────────────────
checkpointer = SqliteSaver.from_conn_string(DB_PATH)
graph = graph_builder.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully.")
print(f"   Checkpoint DB : {DB_PATH}")
print(f"   Nodes         : {list(graph.nodes)}")


## 📊 Step 9 — Visualise Graph (Optional)

In [ ]:
try:
    from IPython.display import Image, display
    img_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(img_bytes))
    print("✅ Graph rendered above.")
except Exception as e:
    print(f"⚠️  PNG render unavailable ({e})")
    print("   Paste the Mermaid source below at https://mermaid.live\n")
    print(graph.get_graph().draw_mermaid())


## 🖥️ Step 10 — HITL Interaction Utilities

- `display_interrupt()` — pretty-print the interrupt payload  
- `collect_human_decision()` — interactive CLI: A / E / R per tool call  
- `run_agent_with_hitl()` — full HITL loop (interactive)


In [ ]:
def display_interrupt(interrupt_value: dict) -> None:
    """Pretty-print the HITL interrupt payload to the CLI."""
    print("\n" + "═" * 65)
    print("  ⏸️   HITL INTERRUPT — HUMAN REVIEW REQUIRED")
    print("═" * 65)
    for tc in interrupt_value.get("pending_tool_calls", []):
        print(f"  [{tc['index']}] Tool    : {tc['tool']}")
        print(f"      Args    : {json.dumps(tc['args'], indent=6)}")
        print(f"      Call ID : {tc['call_id']}")
        print()
    print("  Options: [A]pprove  [E]dit  [R]eject")
    print("═" * 65 + "\n")


def collect_human_decision(interrupt_value: dict) -> dict:
    """
    Interactive CLI loop — collect approve / edit / reject for each tool call.
    Returns the resume payload: {'decisions': [...]}.
    """
    tool_calls = interrupt_value.get("pending_tool_calls", [])
    decisions  = []

    for tc in tool_calls:
        print(f"\n🔧 Tool [{tc['index']}]: {tc['tool']}")
        print(f"   Args: {json.dumps(tc['args'])}")

        while True:
            choice = input("   Decision [A/E/R]: ").strip().upper()

            if choice == "A":
                decisions.append({"action": "approve"})
                print("   ✅ Approved")
                break

            elif choice == "E":
                print(f"   Current args: {json.dumps(tc['args'])}")
                raw = input("   Enter new args as JSON dict: ").strip()
                try:
                    new_args = json.loads(raw)
                    decisions.append({"action": "edit", "edited_args": new_args})
                    print(f"   ✏️  Edited → {new_args}")
                    break
                except json.JSONDecodeError:
                    print("   ⚠️  Invalid JSON — try again.")

            elif choice == "R":
                reason = input("   Rejection reason: ").strip() or "Rejected by human."
                decisions.append({"action": "reject", "reason": reason})
                print(f"   ❌ Rejected — {reason}")
                break

            else:
                print("   ⚠️  Enter A, E, or R")

    return {"decisions": decisions}


def run_agent_with_hitl(query: str, thread_id: str | None = None) -> None:
    """
    Main entry point — runs the agent with full interactive HITL loop.

    Lifecycle:
      1. Stream graph → pause at interrupt
      2. Display tool calls → collect human decision
      3. Resume with Command(resume=...)
      4. Repeat until final answer
    """
    thread_id = thread_id or str(uuid.uuid4())
    config    = {"configurable": {"thread_id": thread_id}}

    print("\n" + "━" * 65)
    print(f"  🔍 Research Agent Pro  │  Thread: {thread_id[:8]}…")
    print(f"  Query: {query}")
    print("━" * 65 + "\n")

    # ── Initial invocation ────────────────────────────────────────────────────
    input_state = {
        "messages":           [HumanMessage(content=query)],
        "pending_tool_calls": [],
        "tool_decisions":     [],
    }

    for chunk in graph.stream(input_state, config=config, stream_mode="values"):
        last = chunk.get("messages", [None])[-1]
        if isinstance(last, AIMessage) and last.content and not last.tool_calls:
            print("\n🤖 Agent:", last.content)

    # ── HITL resume loop ──────────────────────────────────────────────────────
    for _ in range(20):  # safety limit: 20 tool-call rounds max
        snapshot = graph.get_state(config)

        # Locate the interrupt payload
        interrupt_payload = None
        for task in snapshot.tasks:
            for itpt in getattr(task, "interrupts", []):
                interrupt_payload = itpt.value
                break
            if interrupt_payload:
                break

        if not interrupt_payload:
            break  # Agent has finished

        # ── Collect human decision ────────────────────────────────────────────
        display_interrupt(interrupt_payload)
        resume_payload = collect_human_decision(interrupt_payload)

        print("\n▶  Resuming from checkpoint…\n")

        # ── Resume the graph ──────────────────────────────────────────────────
        for chunk in graph.stream(
            Command(resume=resume_payload),
            config=config,
            stream_mode="values",
        ):
            last = chunk.get("messages", [None])[-1]
            if isinstance(last, AIMessage) and last.content and not last.tool_calls:
                print("\n🤖 Agent:", last.content)

    print("\n" + "━" * 65)
    print("  ✅ Research complete.")
    print("━" * 65 + "\n")


print("✅ HITL utilities ready:")
print("   • display_interrupt(payload)")
print("   • collect_human_decision(payload) → resume dict")
print("   • run_agent_with_hitl(query, thread_id)")


## 🤖 Step 11 — Automated Decision Helpers (Testing / CI)
Use these when you want programmatic control — no human prompts required.


In [ ]:
def auto_approve_all(interrupt_value: dict) -> dict:
    """Approve every pending tool call — useful for testing."""
    n = len(interrupt_value.get("pending_tool_calls", []))
    return {"decisions": [{"action": "approve"} for _ in range(n)]}


def auto_reject_all(interrupt_value: dict, reason: str = "Rejected in test mode") -> dict:
    """Reject every pending tool call — useful for testing guardrails."""
    n = len(interrupt_value.get("pending_tool_calls", []))
    return {"decisions": [{"action": "reject", "reason": reason} for _ in range(n)]}


def run_agent_auto(
    query: str,
    decision_fn=auto_approve_all,
    thread_id: str | None = None,
) -> str:
    """
    Run the agent with a programmatic decision function (no human input).
    Returns the final answer string.

    Usage:
        answer = run_agent_auto("What is NDVI?", decision_fn=auto_approve_all)
        answer = run_agent_auto("...", decision_fn=auto_reject_all)
    """
    thread_id = thread_id or str(uuid.uuid4())
    config    = {"configurable": {"thread_id": thread_id}}

    input_state = {
        "messages":           [HumanMessage(content=query)],
        "pending_tool_calls": [],
        "tool_decisions":     [],
    }

    final_answer = ""

    for chunk in graph.stream(input_state, config=config, stream_mode="values"):
        last = chunk.get("messages", [None])[-1]
        if isinstance(last, AIMessage) and last.content and not last.tool_calls:
            final_answer = last.content

    # HITL resume loop (programmatic)
    for _ in range(20):
        snapshot = graph.get_state(config)
        interrupt_payload = None
        for task in snapshot.tasks:
            for itpt in getattr(task, "interrupts", []):
                interrupt_payload = itpt.value
                break
            if interrupt_payload:
                break

        if not interrupt_payload:
            break

        print(f"   [AUTO] Decision for: {[tc['tool'] for tc in interrupt_payload.get('pending_tool_calls', [])]}")
        resume_payload = decision_fn(interrupt_payload)

        for chunk in graph.stream(
            Command(resume=resume_payload),
            config=config,
            stream_mode="values",
        ):
            last = chunk.get("messages", [None])[-1]
            if isinstance(last, AIMessage) and last.content and not last.tool_calls:
                final_answer = last.content

    return final_answer


print("✅ Automated helpers ready:")
print("   • auto_approve_all(payload)              → approve all")
print("   • auto_reject_all(payload, reason)       → reject all")
print("   • run_agent_auto(query, decision_fn)     → non-interactive run")


## 🚀 Step 12 — Run the Agent (Interactive HITL)
Execute the cell below to start a live research session.  
At each tool call, you will see a prompt: **[A]pprove  [E]dit  [R]eject**


In [ ]:
# ── Interactive HITL session ─────────────────────────────────────────────────
# Change the query to any research question!

query = "What are the latest advances in remote sensing for urban heat island mapping?"

run_agent_with_hitl(query)


## 🧪 Step 13 — Automated Test (No Human Input)
Run in auto-approve mode — no prompts, useful for demos and notebooks.


In [ ]:
# ── Auto-approve mode ────────────────────────────────────────────────────────
test_query = "Explain groundwater potential zone mapping using GIS and AHP."

answer = run_agent_auto(test_query, decision_fn=auto_approve_all)

print("\n📝 Final Answer:\n")
print(answer)


## 🔍 Step 14 — Inspect Checkpoint State
Retrieve the full conversation history stored in SqliteSaver for any thread.


In [ ]:
def inspect_thread(thread_id: str) -> None:
    """Print all messages stored in a checkpoint thread."""
    config = {"configurable": {"thread_id": thread_id}}
    state  = graph.get_state(config)

    print(f"\n📦 Checkpoint State — Thread: {thread_id[:8]}…")
    print("─" * 60)
    for i, msg in enumerate(state.values.get("messages", [])):
        role    = type(msg).__name__.replace("Message", "")
        content = str(msg.content)[:200].replace("\n", " ")
        print(f"  [{i+1:02d}] {role:12s} : {content}")
    print("─" * 60)
    print(f"  Next nodes : {state.next}")
    print()


# ── List threads from SQLite directly ────────────────────────────────────────
import sqlite3

def list_threads(db_path: str = DB_PATH, limit: int = 10) -> None:
    """List all checkpoint thread IDs stored in the database."""
    try:
        conn = sqlite3.connect(db_path)
        cur  = conn.execute(
            "SELECT DISTINCT thread_id FROM checkpoints ORDER BY rowid DESC LIMIT ?",
            (limit,)
        )
        rows = cur.fetchall()
        conn.close()
        print(f"\n📂 Recent threads in '{db_path}':")
        for row in rows:
            print(f"   • {row[0]}")
    except Exception as e:
        print(f"⚠️  Could not read DB: {e}")


list_threads()
print()
print("💡 Usage:")
print('   inspect_thread("paste-thread-id-here")')


## 📋 Summary — Architecture & Key Rules

```
╔══════════════════════════════════════════════════════════════╗
║         RESEARCH AGENT PRO — HITL EXECUTION FLOW            ║
╠══════════════════════════════════════════════════════════════╣
║  1  User sends query        →  HumanMessage injected        ║
║  2  agent_node fires        →  LLM picks best tool          ║
║  3  should_continue router  →  tool_calls? → hitl node      ║
║  4  hitl_node: interrupt()  →  state saved to SQLite        ║
║  5  Human reviews           →  display_interrupt()          ║
║  6  Human decides           →  A / E / R per tool call      ║
║  7  Command(resume=...)     →  graph restored from ckpt     ║
║  8  tool_executor_node      →  run / edit / skip tool       ║
║  9  Loop back to agent_node →  LLM processes result         ║
║  10 No more tool_calls      →  final AIMessage → END        ║
╚══════════════════════════════════════════════════════════════╝
```

### ⚠️ Critical Rules for interrupt()
- **Never wrap `interrupt()` in try/except** — it raises `GraphInterrupt` internally
- **Reuse `thread_id`** to resume a paused session; new `thread_id` = fresh session
- **JSON-serialisable payloads only** — no functions, lambdas, or class instances
- **Side effects before `interrupt()` must be idempotent** — the node re-runs on resume
- **Keep interrupt call order consistent** — LangGraph matches resume values by index

### 📦 uv add (quick reference)
```bash
uv add langchain langchain-core langchain-ollama langchain-community langgraph duckduckgo-search wikipedia arxiv
```
